In [1]:
import sys
sys.path.insert(0, "..")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from src.pdm.config import load_config
from src.pdm.data_loader import load_raw_data, build_base_dataframe
from src.pdm.features import build_feature_matrix
from src.pdm.labels import make_labels
from src.pdm.model import pass1_feature_selection, tune_with_optuna, pass2_shap_feature_selection, train_final_model

sns.set_theme(style="whitegrid")
cfg = load_config("../config.yaml")
raw = load_raw_data("../data/raw/")
print(f"Config loaded: horizon={cfg.labeling.horizon_hours}h, n_trials={cfg.model.tuning.n_trials}")

Config loaded: horizon=12h, n_trials=50


In [2]:
base_df = build_base_dataframe(raw)
feature_df = build_feature_matrix(base_df, raw["errors"], raw["maint"], cfg.features.window_hours)
feature_df = feature_df.sort_values("datetime").reset_index(drop=True)
feature_df["label"] = make_labels(feature_df, raw["failures"], cfg.labeling.horizon_hours).values

split_idx = int(len(feature_df) * (1 - cfg.evaluation.test_size_pct))
train_df = feature_df.iloc[:split_idx]
test_df = feature_df.iloc[split_idx:]

NON_FEAT = ["machineID", "datetime", "label"]
X_train = train_df.drop(columns=NON_FEAT, errors="ignore")
y_train = train_df["label"]
X_test = test_df.drop(columns=NON_FEAT, errors="ignore")
y_test = test_df["label"]
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
print(f"Features before selection: {X_train.shape[1]}")
print(f"Label rate \u2014 train: {y_train.mean():.3%} | test: {y_test.mean():.3%}")

Train: 700,880 rows | Test: 175,220 rows
Features before selection: 76
Label rate — train: 1.005% | test: 0.897%


In [ ]:
tscv = TimeSeriesSplit(n_splits=cfg.model.n_cv_folds)
fig, ax = plt.subplots(figsize=(12, 4))
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    ax.barh(fold, len(train_idx), left=0, height=0.4, color="steelblue", alpha=0.7, label="Train" if fold==0 else "")
    ax.barh(fold, len(val_idx), left=len(train_idx), height=0.4, color="coral", alpha=0.9, label="Val" if fold==0 else "")
ax.set_xlabel("Number of samples")
ax.set_ylabel("Fold")
ax.set_title(f"TimeSeriesSplit \u2014 {cfg.model.n_cv_folds} Expanding Folds (no future leakage)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
selected_p1 = pass1_feature_selection(X_train, y_train)
X_train_p1 = X_train[selected_p1]
X_test_p1 = X_test[selected_p1]
print(f"Pass 1: {X_train.shape[1]} \u2192 {len(selected_p1)} features")
print(f"Dropped {X_train.shape[1]-len(selected_p1)} zero-importance features")

In [ ]:
print(f"Running Optuna ({cfg.model.tuning.n_trials} trials, {cfg.model.n_cv_folds}-fold CV)...")
best_params, study = tune_with_optuna(
    X_train_p1, y_train,
    cfg.model.tuning.search_space,
    cfg.model.tuning.n_trials,
    cfg.model.n_cv_folds,
)
print(f"Best CV AUC-PR: {study.best_value:.4f}")
print(f"Best params:\n{json.dumps(best_params, indent=2)}")

trials_df = study.trials_dataframe()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trials_df.index, trials_df["value"], "o-", alpha=0.5, markersize=4, color="steelblue")
ax.axhline(study.best_value, color="red", linestyle="--", label=f"Best: {study.best_value:.4f}")
ax.set_xlabel("Trial")
ax.set_ylabel("AUC-PR")
ax.set_title("Optuna: Hyperparameter Search Convergence")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import shap
interim_model = train_final_model(X_train_p1, y_train, best_params)
selected_p2 = pass2_shap_feature_selection(interim_model, X_train_p1, cfg.features.top_n_features)
X_train_final = X_train_p1[selected_p2]
X_test_final = X_test_p1[selected_p2]
print(f"Pass 2: {len(selected_p1)} \u2192 {len(selected_p2)} features (SHAP top-{cfg.features.top_n_features})")
print(f"\nTop features: {selected_p2[:5]}")

explainer = shap.TreeExplainer(interim_model)
shap_values = explainer.shap_values(X_train_p1)
shap.summary_plot(shap_values, X_train_p1, max_display=20, show=True)

In [ ]:
model = train_final_model(X_train_final, y_train, best_params)
print(f"Final model trained on {len(selected_p2)} features.")
print(f"Top 5 features: {selected_p2[:5]}")